# Exploración: SEACE - Antecedentes de Proveedores (Pentaho BI)

- **Portal origen:** `https://bi.seace.gob.pe/pentaho`
- **Mecanismo:** Motor analítico CDA (`Community Data Access`) vía POST HTTP.
- **Endpoint:** `/pentaho/plugin/cda/api/doQuery?userid=public&password=key`
- **Información que contiene:**
  - Presencia y vigencia en el Registro Nacional de Proveedores (RNP)[cite: 9, 10].
  - Composición societaria actual: socios, representantes y porcentaje accionario[cite: 10].
  - Historial de contrataciones públicas y órdenes de compra/servicio (desde 2008)[cite: 10, 11].
- **Identificadores CDA utilizados:**
  - `QUERY_R_F2_new`: Accionistas y representantes legales.
  - `QUERY_R_F3`: Vigencias por capítulo (bienes / servicios)[cite: 10].
  - `QUERY_R_F5_D_DASH_PROV_RESUMEN`: Historial de montos adjudicados y procesos[cite: 10, 11].

In [22]:
import asyncio
import httpx
import json
from datetime import datetime, timezone
from typing import Dict, Any, List

# Criterio a evaluar (RUC o DNI)
PARAMETRO_BUSQUEDA = "20602102344"

# Configuración base del conector CDA
URL_CDA_AUTH = "https://bi.seace.gob.pe/pentaho/plugin/cda/api/doQuery?userid=public&password=key"
HEADERS_SEACE = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "X-Requested-With": "XMLHttpRequest",
    "Origin": "https://bi.seace.gob.pe",
    "Referer": "https://bi.seace.gob.pe/pentaho/api/repos/:public:ANTECEDENTES_PROVEEDORES:ANTECEDENTES_PROVEEDORES.wcdf/generatedContent?userid=public&password=key"
}
QUERIES = {
    "socios": "QUERY_R_F2_new",
    "vigencias": "QUERY_R_F3",
    "adjudicaciones": "QUERY_R_F5_D_DASH_PROV_RESUMEN"
}

print(f"Parámetro configurado: {PARAMETRO_BUSQUEDA}")

Parámetro configurado: 20602102344


In [23]:
def matricial_a_dict(data_json: dict) -> List[Dict[str, Any]]:
    if not data_json or "resultset" not in data_json:
        return []
    columnas = [col["colName"].strip().upper() for col in data_json.get("metadata", [])]
    return [dict(zip(columnas, fila)) for fila in data_json.get("resultset", [])]

async def pedir_cda(client: httpx.AsyncClient, ruc: str, query_id: str) -> Dict[str, Any]:
    form_data = {
        "path": "/public/ANTECEDENTES_PROVEEDORES/ANTECEDENTES_PROVEEDORES.cda",
        "dataAccessId": query_id,
        "paramPARAM_RUC": ruc.strip(),
        "outputIndexId": "1",
        "pageSize": "0",
        "pageStart": "0",
        "paramsearchBox": ""
    }
    try:
        resp = await client.post(URL_CDA_AUTH, data=form_data, timeout=25.0)
        return {"exito": resp.status_code == 200, "raw": resp.json() if resp.status_code == 200 else None}
    except Exception as exc:
        return {"exito": False, "raw": None, "error": str(exc)}

async def extraer_seace(ruc: str) -> Dict[str, Any]:
    inicio = datetime.now(timezone.utc)
    async with httpx.AsyncClient(headers=HEADERS_SEACE) as client:
        res_soc, res_vig, res_adj = await asyncio.gather(
            pedir_cda(client, ruc, QUERIES["socios"]),
            pedir_cda(client, ruc, QUERIES["vigencias"]),
            pedir_cda(client, ruc, QUERIES["adjudicaciones"]),
            return_exceptions=True
        )
    duracion = int((datetime.now(timezone.utc) - inicio).total_seconds() * 1000)

    # Parseo de datos normalizados
    socios = [
        {
            "tipo_relacion": r.get("TIPO"),
            "documento": str(r.get("NRO_DOC", "")).strip(),
            "nombre": r.get("NOMBRE O RAZON SOCIAL"),
            "acciones": float(r.get("ACCIONES", 0.0) or 0.0)
        }
        for r in matricial_a_dict(res_soc.get("raw") if isinstance(res_soc, dict) else None)
    ]

    vigencias = [
        {
            "capitulo": r.get("CAPITULORNP"),
            "estado": r.get("ESTADO2"),
            "fecha_inicio": r.get("FINICIO"),
            "fecha_fin": r.get("FFIN"),
            "departamento": r.get("DEPARTAMENTOOEE")
        }
        for r in matricial_a_dict(res_vig.get("raw") if isinstance(res_vig, dict) else None)
    ]

    adjudicaciones = [
        {
            "periodo": str(r.get("PERIODO")),
            "modalidad": r.get("MODALIDAD"),
            "entidad": r.get("ENTIDAD"),
            "monto": float(r.get("MONTO", 0.0) or 0.0),
            "procesos": int(r.get("CONVOCATORIAS", 0) or 0)
        }
        for r in matricial_a_dict(res_adj.get("raw") if isinstance(res_adj, dict) else None)
    ]

    return {
        "auditoria": {
            "criterio": ruc,
            "fuente": "SEACE_PENTAHO_CDA",
            "duracion_ms": duracion,
            "estado": "EXITOSO" if (socios or vigencias or adjudicaciones) else "VACIO"
        },
        "datos": {
            "ruc": ruc,
            "vigencias": vigencias,
            "socios": socios,
            "adjudicaciones": adjudicaciones
        },
        "raw": {
            "socios": res_soc.get("raw") if isinstance(res_soc, dict) else None,
            "vigencias": res_vig.get("raw") if isinstance(res_vig, dict) else None,
            "adjudicaciones": res_adj.get("raw") if isinstance(res_adj, dict) else None
        }
    }

resultado = await extraer_seace(PARAMETRO_BUSQUEDA)
print(f"Estado: {resultado['auditoria']['estado']} | Duración: {resultado['auditoria']['duracion_ms']} ms")

Estado: EXITOSO | Duración: 649 ms


In [24]:
print("=================== VIGENCIAS RNP ===================")
for v in resultado["datos"]["vigencias"]:
    print(f"[{v['estado']}] {v['capitulo']} | Del {v['fecha_inicio']} al {v['fecha_fin']} ({v['departamento']})")

print("\n============= COMPOSICIÓN SOCIETARIA =============")
for s in resultado["datos"]["socios"]:
    print(f"[{s['tipo_relacion']}] {s['nombre']} | Doc: {s['documento']} | {s['acciones']}%")

print("\n============ HISTORIAL ADJUDICACIONES ============")
for a in resultado["datos"]["adjudicaciones"]:
    print(f"Año {a['periodo']} | {a['entidad']} | S/ {a['monto']:,.2f} ({a['procesos']} convocatorias)")

=================== VIGENCIAS RNP ===================
[NO VIGENTE] Proveedor de Servicios | Del 07/06/2018 al 13/06/2025 (CUSCO)
[NO VIGENTE] Proveedor de Bienes | Del 07/06/2018 al 13/06/2025 (CUSCO)

============= COMPOSICIÓN SOCIETARIA =============
[ACCIONISTA] RUIZ AUCCAPIÑA ELISBAN | Doc: 43453374 | 100.0%
[ORG. ADMINISTRACION] RUIZ AUCCAPIÑA ELISBAN | Doc: 43453374 | 0.0%
[REPRESENTANTE] RUIZ AUCCAPIÑA ELISBAN | Doc: 43453374 | 0.0%

============ HISTORIAL ADJUDICACIONES ============
Año 2025 | MUNICIPALIDAD PROVINCIAL DE LA CONVENCION - SANTA ANA | S/ 134,778.07 (2 convocatorias)
Año 2025 | MUNICIPALIDAD PROVINCIAL DE LA CONVENCION - SANTA ANA | S/ 110,022.91 (1 convocatorias)
Año 2024 | MUNICIPALIDAD PROVINCIAL DEL CUSCO | S/ 375,741.00 (1 convocatorias)
Año 2024 | MUNICIPALIDAD DISTRITAL DE WANCHAQ | S/ 229,054.00 (5 convocatorias)
Año 2024 | MUNICIPALIDAD PROVINCIAL DEL CUSCO | S/ 224,841.00 (3 convocatorias)
Año 2024 | MUNICIPALIDAD DISTRITAL DE WANCHAQ | S/ 215,740.08 (1 c